In [1]:
import pandas as pd
# On recharge le fichier CSV généré à l'étape précédente
df = pd.read_csv("churn_data.csv")
df.head() # affiche les 5 premières lignes, pour un contrôle visuel rapide

,age,salaire,abonnement,anciennete,consommation,churn
0,56,4543,Premium,12,77,0
1,69,6508,Premium,0,83,0
2,46,5270,Standard,9,21,0
3,32,5999,Premium,4,74,0
4,60,2670,Premium,11,26,0


In [2]:
df.shape # (nombre de lignes, nombre de colonnes) -> doit afficher (500, 6)

(500, 6)

In [3]:
df.describe

<bound method NDFrame.describe of      age  salaire abonnement  anciennete  consommation  churn
0     56     4543    Premium          12            77      0
1     69     6508    Premium           0            83      0
2     46     5270   Standard           9            21      0
3     32     5999    Premium           4            74      0
4     60     2670    Premium          11            26      0
..   ...      ...        ...         ...           ...    ...
495   65     5821    Basique          12             1      0
496   42     6631    Basique          10            48      0
497   57     6178    Basique           3            76      0
498   62     6326    Premium           1             3      1
499   18     6322   Standard          14            89      0

[500 rows x 6 columns]>

In [4]:
df.isnull().sum()

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [5]:
df["churn"].value_counts() 

churn
0    481
1     19
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

# ============================================================
# 1. Conversion de la colonne « abonnement »
# ============================================================
# Les algorithmes de Machine Learning travaillent avec des nombres.
# On convertit donc les catégories textuelles en valeurs numériques.

df["abonnement"] = df["abonnement"].map({
    "Basique": 0,
    "Standard": 1,
    "Premium": 2
})

# ============================================================
# 2. Définition des variables explicatives (features)
# ============================================================
X = df[
    [
        "age",
        "salaire",
        "abonnement",
        "anciennete",
        "consommation"
    ]
]

# ============================================================
# 3. Définition de la variable cible (target)
# ============================================================
y = df["churn"]

# ============================================================
# 4. Séparation des données en Train / Test
# - 80 % pour entraîner le modèle
# - 20 % pour évaluer le modèle sur des données non utilisées
#   pendant l'entraînement
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,   # 20 % des données réservées au test
    random_state=42 # Garantir la reproductibilité de la séparation
)

In [7]:
X_train.shape

(400, 5)

In [8]:
X_test.shape

(100, 5)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ============================================================
# 1. Création du modèle
# ============================================================
# On crée une forêt aléatoire composée de 100 arbres de décision.
# max_depth=5 limite la profondeur des arbres afin de réduire
# le risque de surapprentissage (overfitting).

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# ============================================================
# 2. Entraînement du modèle
# ============================================================
# .fit() déclenche l'apprentissage.
# Le modèle analyse X_train et y_train pour apprendre
# les relations entre les variables explicatives et la cible.

model.fit(X_train, y_train)

# ============================================================
# 3. Prédiction sur les données de test
# ============================================================
# On utilise les données de test, qui n'ont pas été utilisées
# pendant l'entraînement, pour évaluer le modèle.

y_pred = model.predict(X_test)

# ============================================================
# 4. Évaluation du modèle
# ============================================================
# Accuracy : proportion de prédictions correctes.

print("Accuracy :", accuracy_score(y_test, y_pred))

# Classification report :
# affiche la précision (precision), le rappel (recall),
# le score F1 (f1-score) et le nombre d'exemples (support)
# pour chaque classe.

print(classification_report(y_test, y_pred))

Accuracy : 0.99
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



In [10]:
import joblib
# joblib.dump() sérialise (transforme en fichier binaire) l'objet "model" entraîné.
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


In [11]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
n_estimators=50, # moins d'arbres qu'à l'essai 1 (100 -> 50)
max_depth=3, # arbres moins profonds qu'à l'essai 1 (5 -> 3)
random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl") # écrase le fichier de l'Essai 1
print("model.pkl écrasé avec l'Essai 2")

Accuracy Essai 2 : 0.99
model.pkl écrasé avec l'Essai 2


In [12]:
# ---------- ESSAI 3 : on retire la colonne "salaire" ----------
X = df[["age", "abonnement", "anciennete", "consommation"]] # salaire retiré volontairement
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 3 (sans salaire) :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl") # écrase le fichier de l'Essai 2
print("model.pkl écrasé avec l'Essai 3")

Accuracy Essai 3 (sans salaire) : 1.0
model.pkl écrasé avec l'Essai 3
